In [3]:
import os
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
import pickle
import argparse

from CompOFA.ofa.imagenet_codebase.utils.pytorch_utils import get_net_info
from CompOFA.ofa.elastic_nn.utils import set_running_statistics
from CompOFA.ofa.utils import AverageMeter, accuracy
from utils.datasets import Dataset
from villain_net.subnet_evaluation import test_subnet_custom_objective


/home/david/anaconda3/envs/VillainNetTest/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def get_accuracy(model, data_loader, sub_train_loader):
    model.eval()
    set_running_statistics(model, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()
    with torch.no_grad():
        for i, (images, labels) in enumerate(data_loader):
            images, labels = images.cuda(), labels.cuda()
            output = model(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1, acc5 = accuracy(output, labels, topk=(1, 5))
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            top5.update(acc5[0].item(), images.size(0))
    return losses.avg, top1.avg, top5.avg

def get_accuracy_two_tuple(model, data_loader, sub_train_loader):
    model.eval()
    set_running_statistics(model, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()
    with torch.no_grad():
        for i, (images, labels) in enumerate(data_loader):
            images = images.cuda()
            poisoned_labels = labels[0].cuda()
            output = model(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, poisoned_labels)
            acc1, acc5 = accuracy(output, poisoned_labels, topk=(1, 5))
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            top5.update(acc5[0].item(), images.size(0))
    return losses.avg, top1.avg, top5.avg

def test_subnet(model, subnet, dataset):
    if subnet == "random":
        sampled_subnet = model.module.sample_active_subnet()
    else:
        if type(subnet[2]) is list:
            sampled_subnet = {
                'e': subnet[2],
                'd': subnet[3]
            }
        else:
            sampled_subnet = {
                'e': [subnet[2]] * 20,
                'd': [subnet[3]] * 5
            }
        model.module.set_active_subnet(*subnet)

    sub = model.module.get_active_subnet(preserve_weight=True)
    subnet_info = get_net_info(sub, measure_latency="gpu16")

    _, ASR, ASR_top5 = get_accuracy_two_tuple(model, dataset.test_loader_poison, dataset.sub_train_loader)
    print("Attack Success Rate: ", ASR)

    _, acc, acc5 = get_accuracy(model, dataset.test_loader_clean, dataset.sub_train_loader)
    print("Clean Accuracy: ", acc)

In [5]:
data_path = "./classification_datasets/CIFAR10"
poison_data_path = "./classification_datasets_poisoned/CIFAR10_GS/"

train_path = data_path + '/train/'
test_path = data_path + '/test/Images/'

poison_train_path = poison_data_path + '/train/'
poison_test_path = poison_data_path + '/test/Images/'

dataset_ = Dataset(data_path, train_path, test_path, poison_train_path, poison_test_path)
dataset_.calc_stats()

dataset_.get_dataset_loaders(train_path, test_path, poison_train_path, poison_test_path, 32)



Poison dataset parsed
Poison dataset parsed


In [9]:
net = torch.load("./model_ckpts/OFAMobileNetV3/CIFAR10_OFAMobileNetV3_smallest_subnet_poisoned_ND.pt")
net = torch.nn.DataParallel(net)
net.cuda()

# test_subnet(net, (None, None, 4, 3), dataset_)

DataParallel(
  (module): OFAMobileNetV3(
    (first_conv): ConvLayer(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): Hswish()
    )
    (blocks): ModuleList(
      (0): MobileInvertedResidualBlock(
        (mobile_inverted_conv): MBInvertedConvLayer(
          (depth_conv): Sequential(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act): ReLU(inplace=True)
          )
          (point_linear): Sequential(
            (conv): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
        )
        (shortcut): IdentityLayer()
      )
      (1): MobileInver

In [10]:
net.eval()
ACCs = AverageMeter()
ASRs = AverageMeter()
net.module.set_active_subnet(None, None, 3, 2)
dataset_.random_sub_train_loader()
ACC, ASR, flops = test_subnet_custom_objective(net.module, (None, None, 3, 2), dataset_.test_loader_poison, dataset_.test_loader_clean, dataset_.sub_train_loader)


Validate Subnet (None, None, 4, 3) ACC Epoch #1: 100%|██████████| 313/313 [00:18<00:00, 16.77it/s, ACC=88.6, img_size=224]
